# **Load Parameters**

In [0]:
import traceback

try:
    catalog = dbutils.widgets.get("catalog")
    silver_schema = dbutils.widgets.get("silver_schema")
    gold_schema = dbutils.widgets.get("gold_schema")
    silver_table = dbutils.widgets.get("silver_table")
    dim_vendor = dbutils.widgets.get("dim_vendor")
    dim_store = dbutils.widgets.get("dim_store")
    dim_product = dbutils.widgets.get("dim_product")
except Exception as e:
    print("Error getting notebook parameters")
    print(traceback.format_exc())
    raise e

## **Creating the Dimension Tables**
This step ensures all dimension tables **[Vendor,Store,Prodcut]** exist in the Gold schema. Using CREATE TABLE IF NOT EXISTS makes this operation idempotent.

In [0]:
try:
    print(f"Ensuring table {catalog}.{gold_schema}.{dim_vendor} exists...")
    create_vendor_query = f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.{dim_vendor} (
      vendor_key BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
      vendor_business_key INT,
      vendor_name STRING,
      saved_date TIMESTAMP
    )
    USING DELTA
    COMMENT 'Vendor Dimension'
    """
    spark.sql(create_vendor_query)
    
    print(f"Ensuring table {catalog}.{gold_schema}.{dim_product} exists...")
    create_product_query = f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.{dim_product} (
      product_key BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
      product_business_key INT,
      item_description STRING,
      pack INT,
      bottle_volume_ml INT,
      category_code INT,
      category_name STRING,
      saved_date TIMESTAMP
    )
    USING DELTA
    COMMENT 'Product Dimension'
    """
    spark.sql(create_product_query)

    print(f"Ensuring table {catalog}.{gold_schema}.{dim_store} exists...")
    create_store_query = f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.{dim_store} (
      store_key BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
      store_business_key INT,
      store_name STRING,
      address STRING,
      city STRING,
      zip_code STRING,
      county STRING,
      saved_date TIMESTAMP 
    )
    USING DELTA
    COMMENT 'Store Dimension'
    """
    spark.sql(create_store_query)
    
    print("All dimension tables are ready.")

except Exception as e:
    print("Error creating dimension tables")
    print(traceback.format_exc())
    raise e

## **Load Dimension Tables (SCD Type 1)**

**Load Vendor Dimension**

In [0]:
try:
    print(f"Merging data into {catalog}.{gold_schema}.{dim_vendor}...")
    
    merge_query_vendor = f"""
    MERGE INTO {catalog}.{gold_schema}.{dim_vendor} AS target
    USING (
      -- Select the most recent record for each vendor
      SELECT * FROM (
        SELECT
          vendor_number,
          vendor_name,
          saved_date,
          ROW_NUMBER() OVER(PARTITION BY vendor_number ORDER BY date DESC) as rn
        FROM {catalog}.{silver_schema}.{silver_table}
        WHERE vendor_number IS NOT NULL
      )
      WHERE rn = 1
    ) AS source
    ON target.vendor_business_key = source.vendor_number
    WHEN MATCHED AND (target.vendor_name <> source.vendor_name) THEN
      -- Update existing vendor name if it changed
      UPDATE SET
        target.vendor_name = source.vendor_name,
        target.saved_date = source.saved_date
    WHEN NOT MATCHED THEN
      -- Insert new vendor
      INSERT (
        vendor_business_key,
        vendor_name,
        saved_date
      )
      VALUES (
        source.vendor_number,
        source.vendor_name,
        source.saved_date
      )
    """
    spark.sql(merge_query_vendor)
    print(f"Vendor dimension load complete.")

except Exception as e:
    print(f"Error loading {dim_vendor}")
    print(traceback.format_exc())
    raise e

**Load Product Dimension**

In [0]:
try:
    print(f"Merging data into {catalog}.{gold_schema}.{dim_product}...")
    
    merge_query_product = f"""
    MERGE INTO {catalog}.{gold_schema}.{dim_product} AS target
    USING (
      -- Select the most recent record for each product
      SELECT * FROM (
        SELECT
          item_number,
          item_description,
          pack,
          bottle_volume_ml,
          category_code,
          category_name,
          saved_date,
          ROW_NUMBER() OVER(PARTITION BY item_number ORDER BY date DESC) as rn
        FROM {catalog}.{silver_schema}.{silver_table}
        WHERE item_number IS NOT NULL
      )
      WHERE rn = 1
    ) AS source
    ON target.product_business_key = source.item_number
    -- Check all relevant attributes for changes to trigger an update
    WHEN MATCHED AND (
         target.item_description <> source.item_description
      OR target.pack <> source.pack
      OR target.bottle_volume_ml <> source.bottle_volume_ml
      OR target.category_code <> source.category_code
      OR target.category_name <> source.category_name
    ) THEN
      -- Update attributes AND the saved_date
      UPDATE SET
        target.item_description = source.item_description,
        target.pack = source.pack,
        target.bottle_volume_ml = source.bottle_volume_ml,
        target.category_code = source.category_code,
        target.category_name = source.category_name,
        target.saved_date = source.saved_date
    WHEN NOT MATCHED THEN
      -- Insert new product with its saved_date
      INSERT (
        product_business_key,
        item_description,
        pack,
        bottle_volume_ml,
        category_code,
        category_name,
        saved_date
      )
      VALUES (
        source.item_number,
        source.item_description,
        source.pack,
        source.bottle_volume_ml,
        source.category_code,
        source.category_name,
        source.saved_date
      )
    """
    spark.sql(merge_query_product)
    print(f"Product dimension load complete.")
    
except Exception as e:
    print(f"Error loading {dim_product}")
    print(traceback.format_exc())
    raise e

**Load Store Dimension**

In [0]:
try:
    print(f"Merging data into {catalog}.{gold_schema}.{dim_store}...")
    
    merge_query_store = f"""
    MERGE INTO {catalog}.{gold_schema}.{dim_store} AS target
    USING (
      -- Select the most recent record for each store
      SELECT * FROM (
        SELECT
          store_number,
          store_name,
          address,
          city,
          zip_code,
          county,
          saved_date,
          ROW_NUMBER() OVER(PARTITION BY store_number ORDER BY date DESC) as rn
        FROM {catalog}.{silver_schema}.{silver_table}
        WHERE store_number IS NOT NULL
      )
      WHERE rn = 1
    ) AS source
    ON target.store_business_key = source.store_number
    -- Check all relevant attributes for changes to trigger an update
    WHEN MATCHED AND (
         target.store_name <> source.store_name
      OR target.address <> source.address
      OR target.city <> source.city
      OR target.zip_code <> source.zip_code
      OR target.county <> source.county
    ) THEN
      -- Update attributes AND the saved_date
      UPDATE SET
        target.store_name = source.store_name,
        target.address = source.address,
        target.city = source.city,
        target.zip_code = source.zip_code,
        target.county = source.county,
        target.saved_date = source.saved_date
    WHEN NOT MATCHED THEN
      -- Insert new store with its saved_date
      INSERT (
        store_business_key,
        store_name,
        address,
        city,
        zip_code,
        county,
        saved_date
      )
      VALUES (
        source.store_number,
        source.store_name,
        source.address,
        source.city,
        source.zip_code,
        source.county,
        source.saved_date
      )
    """
    spark.sql(merge_query_store)
    print(f"Store dimension load complete.")
    
except Exception as e:
    print(f"Error loading {dim_store}")
    print(traceback.format_exc())
    raise e

In [0]:
dbutils.notebook.exit("Gold dimension layer processing complete.")